In [1]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: C:\Users\HoangHoanh\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [6]:
import sys
import pandas as pd
 
src, dst = sys.argv[1], sys.argv[2]
SHEET = "Inventory data"
TEXT_COLS = ["Sku_name", "Sku_code", "shop_account", "Vendor", "Brand"]
UPPER_COLS = ["shop_account", "Vendor", "Brand"]   # cột phân loại -> thống nhất chữ HOA
 
# 1. Đọc (giữ Sku_code dạng chuỗi để không mất số 0 đầu)
df = pd.read_excel(src, sheet_name=SHEET, dtype={"Sku_code": str})
n0 = len(df)
 
# 2. Chuẩn hóa tên cột (gộp mọi khoảng trắng/xuống dòng thành 1 dấu cách)
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()
 
# 3. Cột chữ: gộp khoảng trắng kép, bỏ khoảng trắng đầu/cuối, chuỗi rỗng -> NaN
for c in TEXT_COLS:
    s = df[c].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    df[c] = s.mask(s == "", pd.NA)
for c in UPPER_COLS:
    df[c] = df[c].str.upper()
 
# 4. Cột số: loại ký tự lạ ('#0.0', '0.0 VND'...) rồi ép về số
num_cols = [c for c in df.columns if c not in TEXT_COLS]
bad = {}
for c in num_cols:
    raw = df[c]
    clean = pd.to_numeric(raw.astype("string").str.replace(r"[^\d.\-]", "", regex=True), errors="coerce")
    bad[c] = int((raw.notna() & ~pd.to_numeric(raw, errors="coerce").notna()).sum())
    df[c] = clean
neg = int((df[num_cols] < 0).sum().sum())
 
# 5. Xóa dòng trùng sau khi đã chuẩn hóa
df = df.drop_duplicates().reset_index(drop=True)
 
# 6. Báo cáo
print(f"Số dòng: {n0} -> {len(df)} (xóa {n0-len(df)} dòng trùng)")
print("Ô số lỗi định dạng đã sửa:", {k: v for k, v in bad.items() if v})
print("Giá trị âm:", neg)
print("Số giá trị phân loại duy nhất:", {c: int(df[c].nunique()) for c in UPPER_COLS})
print("Giá trị thiếu theo cột:\n", df.isnull().sum().to_string())
 
# 7. Lưu (chỉ sheet đã làm sạch)
df.to_excel(dst, index=False)

FileNotFoundError: [Errno 2] No such file or directory: '-f'

In [10]:
import pandas as pd, numpy as np, re, os
from pathlib import Path
import pandas as pd

p = Path("data1_cleaned_all.xlsx")
out = Path("data1_cleaned_final.xlsx")

if not p.exists():
    raise FileNotFoundError(
        f"Khong tim thay file: {p.resolve()}. "
        "Hay dat file Excel cung thu muc voi notebook."
    )

xls = pd.ExcelFile(p)
dfs={s:pd.read_excel(p,sheet_name=s) for s in xls.sheet_names}

def clean_text_df(df):
    df=df.copy()
    df.columns=[re.sub(r'\s+',' ',str(c)).strip() for c in df.columns]
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c]):
            df[c]=df[c].map(lambda x: re.sub(r'\s+',' ',x).strip() if isinstance(x,str) else x)
            df[c]=df[c].replace(r'^\s*$',np.nan,regex=True)
    return df

for s in dfs: dfs[s]=clean_text_df(dfs[s])

# Loại bỏ các hàng trùng lặp hoàn toàn
summary=[]
missing=[]
duplicate_keys=[]
for s,df in dfs.items():
    n0=len(df)
    df=df.drop_duplicates().reset_index(drop=True)
    exact_removed=n0-len(df)
    summary.append({"Sheet":s,"Rows before":n0,"Rows after exact-duplicate removal":len(df),"Exact duplicate rows removed":exact_removed})
    for c in df.columns:
        missing.append({"Sheet":s,"Column":c,"Missing values":int(df[c].isna().sum()),"Missing percentage (%)":round(df[c].isna().mean()*100,2)})
    dfs[s]=df

# Định dạng các cột
for s, cols in {
    "Inventory data":["Tiki","Lazada","Shopee","TikTok","Sendo","Tiki demand per month","Lazada demand per month","Shopee demand per month","TikTok demand per month","Sendo demand per month","DOITarget (days)"],
    "SKU Masterdata":["Sale Price","Base Cost"],
    "VendorMasterdata":["Payment Terms (per week)"]
}.items():
    for c in cols:
        if c in dfs[s]:
            dfs[s][c]=pd.to_numeric(dfs[s][c],errors="coerce")

# Build unique, unambiguous lookup maps from SKU masterdata
sku=dfs["SKU Masterdata"].copy()
# normalize keys
def norm_key(x):
    if pd.isna(x): return np.nan
    return str(x).strip().casefold()
sku["_key"]=sku["Sku Code"].map(norm_key)
lookup_cols={"Sku Name":"Sku_name","Primary Vendor":"Vendor","Brand":"Brand"}
sku_maps={}
for src,dst in lookup_cols.items():
    valid=sku.dropna(subset=["_key",src])
    counts=valid.groupby("_key")[src].nunique()
    unique_keys=set(counts[counts==1].index)
    sku_maps[dst]=valid[valid["_key"].isin(unique_keys)].drop_duplicates("_key").set_index("_key")[src].to_dict()

inv=dfs["Inventory data"].copy()
if "Sku_code" in inv.columns:
    inv["_key"]=inv["Sku_code"].map(norm_key)
    for dst,mp in sku_maps.items():
        if dst in inv.columns:
            fillmask=inv[dst].isna() & inv["_key"].notna()
            inv.loc[fillmask,dst]=inv.loc[fillmask,"_key"].map(mp)
    inv=inv.drop(columns=["_key"])
dfs["Inventory data"]=inv

# Build vendor lookup maps where keys are unambiguous
vendor=dfs["VendorMasterdata"].copy()
for keycol in ["SHOP_ACCOUNT","VENDOR","BRAND"]:
    if keycol in vendor.columns:
        vendor["_key_"+keycol]=vendor[keycol].map(norm_key)
# Fill vendor master missing descriptive fields using unique vendor/brand/account matches, only exact unambiguous lookup
vm=vendor.copy()
for target in ["Vendor Name","Shipping Adress","Payment Terms (per week)"]:
    if target not in vm.columns: continue
    for keycol in ["SHOP_ACCOUNT","VENDOR","BRAND"]:
        k="_key_"+keycol
        valid=vm.dropna(subset=[k,target])
        counts=valid.groupby(k)[target].nunique()
        unique=set(counts[counts==1].index)
        mp=valid[valid[k].isin(unique)].drop_duplicates(k).set_index(k)[target].to_dict()
        mask=vm[target].isna() & vm[k].notna()
        vm.loc[mask,target]=vm.loc[mask,k].map(mp)
vendor=vm.drop(columns=[c for c in vm.columns if c.startswith("_key_")])
dfs["VendorMasterdata"]=vendor

# Create key duplicate reports for SKU and vendor keys, don't auto-delete ambiguous key duplicates
for s,keycol in [("SKU Masterdata","Sku Code"),("VendorMasterdata","SHOP_ACCOUNT")]:
    df=dfs[s]
    if keycol in df.columns:
        keys=df[keycol].dropna().map(norm_key)
        dup_keys=keys[keys.duplicated(keep=False)].unique()
        for k in dup_keys:
            subset=df[df[keycol].map(norm_key)==k]
            duplicate_keys.append({"Sheet":s,"Key column":keycol,"Key value":str(subset.iloc[0][keycol]),"Rows":len(subset),"Distinct rows":len(subset.drop_duplicates())})

# recalc missing report post lookup
missing=[]
for s,df in dfs.items():
    for c in df.columns:
        missing.append({"Sheet":s,"Column":c,"Missing values":int(df[c].isna().sum()),"Missing percentage (%)":round(df[c].isna().mean()*100,2)})

# Final summary after lookup
summary=[]
for s,df in dfs.items():
    summary.append({"Sheet":s,"Rows":len(df),"Columns":len(df.columns),"Missing cells":int(df.isna().sum().sum()),"Exact duplicate rows remaining":int(df.duplicated().sum())})

with pd.ExcelWriter(out,engine="openpyxl") as writer:
    for s,df in dfs.items():
        df.to_excel(writer,sheet_name=s[:31],index=False)
    pd.DataFrame(summary).to_excel(writer,sheet_name="Cleaning Summary",index=False)
    pd.DataFrame(missing).to_excel(writer,sheet_name="Missing Values",index=False)
    pd.DataFrame(duplicate_keys,columns=["Sheet","Key column","Key value","Rows","Distinct rows"]).to_excel(writer,sheet_name="Duplicate Keys",index=False)
print(out, os.path.getsize(out))
print(pd.DataFrame(summary).to_string(index=False))


data1_cleaned_final.xlsx 1628584
           Sheet  Rows  Columns  Missing cells  Exact duplicate rows remaining
  Inventory data  2245       16           1614                               0
  SKU Masterdata 16619        7           5973                               0
VendorMasterdata    81        6             16                               0
